# cr-sim on Colab

Training a Clash Royale agent on a GPU runtime.

**Read this before running anything, because it decides whether Colab is worth using at all.**

This workload is bound by *simulating battles in Python*, not by the neural network. Measured on the
development laptop: about 90% of a decision is engine ticks, and the network is 40-50% of the parent
process's time only once environments are spread across worker processes.

Free Colab gives **2 vCPUs**. The laptop has 8. So a T4 will accelerate the part that is not the
bottleneck while quartering the part that is, and a free runtime may well be **slower** than a laptop.

Where Colab does win:

- **Sessions in parallel.** Generating demonstrations and sweeping hyperparameters are embarrassingly
  parallel. Several runtimes doing that beat one laptop easily.
- **A high-CPU runtime.** Colab Pro's A100 and TPU runtimes come with far more vCPUs, which is the
  number that matters here.

Cell 2 measures it rather than guessing. Run it before committing to a long job.

## 1. What runtime is this?

In [ ]:
import os, subprocess, sys
import multiprocessing as mp

print('python  ', sys.version.split()[0])
print('cpus    ', mp.cpu_count(), '(the number that matters here)')
try:
    import torch
    print('torch   ', torch.__version__)
    print('cuda    ', torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except ImportError:
    print('torch    not installed yet')
print()
print(subprocess.run(['free','-h'], capture_output=True, text=True).stdout or '')

## 2. Install

The repo is public. `data_cache/` is not in it -- the game data comes from an APK you supply, which
the next cell handles.

In [ ]:
!git clone --depth 1 https://github.com/evansuckedatlife/cr-sim.git 2>/dev/null || (cd cr-sim && git pull)
%cd /content/cr-sim
!pip install -q -e . numpy gymnasium
import sys; sys.path.insert(0, '/content/cr-sim')
print('installed')

## 3. Game data

The engine is built from Supercell's own shipped tables, which are not redistributed here. Two ways in,
and the Drive one is worth the two minutes because a Colab runtime is wiped every time it disconnects.

**Once, from your machine:** copy the `data_cache` folder into your Drive at `MyDrive/cr-sim/data_cache`.

**Or:** upload a Clash Royale APK (or a zip of the split APKs) when prompted below.

In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/cr-sim')
LOCAL = Path('/content/cr-sim/data_cache')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('no Drive:', exc)

if (DRIVE / 'data_cache' / 'csv_logic').is_dir():
    LOCAL.parent.mkdir(parents=True, exist_ok=True)
    if not LOCAL.exists():
        os.symlink(DRIVE / 'data_cache', LOCAL)
    print('using the copy in Drive')
else:
    print('No data_cache in Drive. Upload an APK and it will be extracted.')
    from google.colab import files
    up = files.upload()
    for name in up:
        !python scripts/extract_apk.py "{name}"

count = len(list((LOCAL / 'csv_logic').glob('*'))) if (LOCAL / 'csv_logic').is_dir() else 0
print(f'{count} logic files')
assert count, 'no game data -- nothing below will run'

## 4. Does this runtime actually beat a laptop?

Measured, not assumed. The laptop baseline is **46 decisions/s** on 8 cores with 6 worker processes.
If this comes back lower, a long run here is slower than running it at home, and the honest use of
Colab is parallel jobs rather than one big one.

In [ ]:
import time, json, subprocess

workers = max(1, mp.cpu_count() - 1)
args = ['python','-m','cr_sim.train.run','--steps','3072','--horizon','128',
        '--envs', str(workers * 2), '--workers', str(workers),
        '--tps','20','--frame-skip','30','--match-seconds','120','--tower-level','5',
        '--reward','projected','--horizon-seconds','3','--opponent','random',
        '--eval-every','0','--save-every','9999','--device','auto',
        '--out','/tmp/bench','--name','bench']
out = subprocess.run(args, capture_output=True, text=True).stdout
print(out[-600:])
rate = None
for line in out.splitlines():
    if '/s' in line:
        rate = float(line.split('/s')[0].split()[-1])
print()
print(f'this runtime : {rate} decisions/s on {mp.cpu_count()} cpus')
print( 'laptop       : 46 decisions/s on 8 cpus')
if rate and rate < 46:
    print('\nSlower than the laptop. Use this runtime for parallel jobs')
    print('(demonstrations, sweeps) rather than one long training run.')

## 5. Train

Checkpoints go to Drive, so a disconnected runtime costs one checkpoint interval rather than the run.
Re-running this cell after a reconnect resumes: it finds the checkpoint and continues from it, with the
optimiser state and step count intact.

Colab disconnects free runtimes after a few hours of inactivity and caps sessions around twelve, so
assume you will need to.

In [ ]:
RUN = 'colab-selfplay'
runs = DRIVE / 'runs'
runs.mkdir(parents=True, exist_ok=True)
resume = (runs / RUN / 'checkpoint.pt').exists()
print('resuming' if resume else 'starting fresh')

cmd = ['python','-m','cr_sim.train.run',
       '--steps','2000000',
       '--envs', str(workers * 2), '--workers', str(workers),
       '--horizon','256','--tps','20','--frame-skip','30',
       '--match-seconds','120','--tower-level','5',
       '--reward','projected','--horizon-seconds','3',
       '--opponent','self','--pool-size','8','--refresh-every','20',
       '--eval-every','20','--ancestor-episodes','30','--save-every','5',
       '--device','auto','--out', str(runs), '--name', RUN]
if resume:
    cmd.append('--resume')

import subprocess, sys
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')

## 6. Tell Discord about it (optional)

The same notifier the laptop uses. A webhook URL is all it needs -- no bot, nothing to log into -- and
it means a Colab run reports to the same channel as everything else.

Get one from **Server Settings > Integrations > Webhooks > New Webhook > Copy Webhook URL**. Paste it
with Colab's secrets panel (the key icon) rather than into the notebook, so it does not end up saved
in the file.

In [ ]:
from google.colab import userdata
import subprocess, os

os.environ['CR_SIM_DISCORD_WEBHOOK'] = userdata.get('CR_SIM_DISCORD_WEBHOOK')
subprocess.Popen(['python','-m','cr_sim.train.notify',
                  '--runs', str(runs), '--every','120'])
print('reporting to Discord every two minutes')

## 7. Bring the result home

Everything already lives in Drive, so this is only for pulling a checkpoint down to look at locally.

```
python scripts/clone_policy.py --demos data_cache/demos --out runs/colab
python -m cr_sim.play.server --policy runs/colab-selfplay/best.pt --tower-level 5
```

In [ ]:
import json
from pathlib import Path

path = runs / RUN / 'metrics.jsonl'
rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
evals = [r for r in rows if 'eval_lift_sd' in r]
print(f"{len(rows)} updates, {rows[-1]['steps']:,} steps, {len(evals)} evaluations")
for r in evals[-8:]:
    print(f"  {r['steps']:>9,}  lift {r['eval_lift_sd']:+.3f}"
          f"  expl var {r.get('explained_variance', float('nan')):+.3f}")